In [0]:
import pandas as pd
import numpy as np

# don't display print or show large data - slow 
# use .limit() to explore data (faster and safer than using the full table)
# filter early 
# convert to Pandas only on smaller results 
# reuse data frames (refrence already loaded data instead of loading over and over)

# spark = getting the data 
# pandas = working with the data 
hno = spark.table("workspace.unochadatasets.hpc_hno_2025")
fts = spark.table("workspace.unochadatasets.fts_requirements_funding_global")

# Get parameter values
crisis_category = dbutils.widgets.get("crisis_category")
include_temporal = dbutils.widgets.get("include_temporal_factor")
demographic_filter = dbutils.widgets.get("demographic_filter")

# Calculate demographic proportions per country
if demographic_filter != "All":
    # Define demographic filters
    if demographic_filter == "Children":
        # Children: under 18 years (age ranges 0-4, 5-9, 10-14, 15-19)
        demo_filter = "Age_range IN ('0-4', '5-9', '10-14', '15-19')"
    elif demographic_filter == "Men":
        # Men: male adults (20+)
        demo_filter = "Gender = 'm' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    elif demographic_filter == "Women":
        # Women: female adults (20+)
        demo_filter = "Gender = 'f' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
    
    # Get demographic population for selected group
    demographic_pop = spark.sql(f"""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as demographic_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE {demo_filter}
    GROUP BY ISO3
    """)
    
    # Get total population per country
    total_pop = spark.sql("""
    SELECT 
        ISO3 as iso3,
        SUM(Population) as total_population
    FROM workspace.unochadatasets.cod_population_admin_0
    WHERE Age_range = 'all' AND Gender = 'all'
    GROUP BY ISO3
    """)
    
    # Calculate proportion
    demo_proportions_spark = demographic_pop.join(total_pop, "iso3", "inner")
    demo_proportions = demo_proportions_spark.toPandas()
    demo_proportions["demographic_proportion"] = (
        demo_proportions["demographic_population"] / demo_proportions["total_population"]
    )
    demo_proportions = demo_proportions[["iso3", "demographic_proportion"]]
else:
    # No demographic filtering - proportion is 1.0 for all countries
    demo_proportions = pd.DataFrame(columns=["iso3", "demographic_proportion"])

# FIX: HNO baseline - AGGREGATE by country to avoid duplicates
hno_base_spark = spark.sql(f"""
SELECT
  `Country ISO3` AS iso3,
  SUM(CAST(`In Need` AS BIGINT)) AS people_in_need
FROM workspace.unochadatasets.hpc_hno_2025
WHERE `Cluster` = '{crisis_category}'
  AND (`Category` IS NULL OR TRIM(`Category`) = '')
GROUP BY `Country ISO3`
""")

hno_base = hno_base_spark.toPandas()
hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])

# Apply demographic proportion to people_in_need
if demographic_filter != "All":
    hno_base = hno_base.merge(demo_proportions, on="iso3", how="left")
    hno_base["demographic_proportion"] = hno_base["demographic_proportion"].fillna(0.5)  # Default if no demographic data
    hno_base["people_in_need"] = (hno_base["people_in_need"] * hno_base["demographic_proportion"]).astype('int64')
    hno_base = hno_base[["iso3", "people_in_need"]]

# FTS baseline - Filter to 2025 to match HNO data
# requirements = how much funding is needed
# funding = how much funding is actually provided
fts_base_spark = spark.sql("""
SELECT
  `countryCode` AS iso3,
  SUM(`requirements`) AS requirements,
  SUM(`funding`) AS funding
FROM workspace.unochadatasets.fts_requirements_funding_global
WHERE `year` = 2025
GROUP BY `countryCode`
""")

fts_base = fts_base_spark.toPandas()
fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])

# Temporal factor calculation (only if enabled)
if include_temporal == "Yes":
    # Get historical funding data (2020-2024) to calculate temporal factor
    fts_historical_spark = spark.sql("""
    SELECT
      `countryCode` AS iso3,
      `year`,
      SUM(`requirements`) AS requirements,
      SUM(`funding`) AS funding
    FROM workspace.unochadatasets.fts_requirements_funding_global
    WHERE `year` BETWEEN 2020 AND 2024
    GROUP BY `countryCode`, `year`
    """)
    
    fts_historical = fts_historical_spark.toPandas()
    fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
    fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
    fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
    fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
    
    # Calculate historical coverage ratios
    fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
    
    # NEW APPROACH: Calculate actual gap scores for each historical year
    # For historical years, we approximate people_in_need from requirements (as we don't have historical HNO data)
    # gap_score_year = requirements × (1 - coverage_ratio)
    fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
    
    # Sum historical gaps per country
    historical_gaps = fts_historical.groupby("iso3")["gap_score_year"].sum().reset_index()
    historical_gaps.columns = ["iso3", "cumulative_historical_gap"]
    
    # Count years for reference
    years_count = fts_historical.groupby("iso3")["year"].count().reset_index()
    years_count.columns = ["iso3", "years_with_data"]
    
    temporal_factor = historical_gaps.merge(years_count, on="iso3", how="left")
else:
    # Create empty temporal factor
    temporal_factor = pd.DataFrame(columns=["iso3", "cumulative_historical_gap", "years_with_data"])

# Merge - based on country 
df = hno_base.merge(fts_base, on="iso3", how="inner")

# Merge temporal factor
df = df.merge(temporal_factor, on="iso3", how="left")
df["cumulative_historical_gap"] = df["cumulative_historical_gap"].fillna(0).astype('float64')
df["years_with_data"] = df["years_with_data"].fillna(0).astype('int64')

# Filter - remove rows where some data is missing 
df = df[(df["people_in_need"] > 0) & (df["requirements"] > 0)].copy()

# Already one row per country from the aggregation in SQL query above
df_country = df.copy()

# Compute baseline
df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"] # how much of the funding was actually provided 
df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1) 

# Base gap score: how many people need help and how underfunded it is
df_country["base_gap_score"] = df_country["people_in_need"] * (1 - df_country["coverage_ratio"])

# Apply temporal multiplier based on parameter setting
if include_temporal == "Yes":
    # NEW FORMULA: Use cumulative historical gaps to calculate multiplier
    # The more unmet need accumulated over years, the higher the multiplier
    # temporal_multiplier = 1 + (average_annual_historical_gap / current_base_gap_score) * weight
    
    # Calculate average annual historical gap
    df_country["avg_annual_historical_gap"] = df_country["cumulative_historical_gap"] / df_country["years_with_data"].replace(0, 1)
    
    # Temporal weight determines how much history influences current score
    TEMPORAL_WEIGHT = 0.5  # 50% weight means if avg historical gap = current gap, you get 1.5x multiplier
    
    # Calculate multiplier: larger historical gaps relative to current gap = higher multiplier
    df_country["temporal_multiplier"] = 1 + (df_country["avg_annual_historical_gap"] / df_country["base_gap_score"].replace(0, 1)) * TEMPORAL_WEIGHT
    
    # Cap the multiplier to avoid extreme values
    df_country["temporal_multiplier"] = df_country["temporal_multiplier"].clip(lower=1.0, upper=3.0)
else:
    # No temporal adjustment - all countries get multiplier of 1.0
    df_country["temporal_multiplier"] = 1.0
    df_country["avg_annual_historical_gap"] = 0.0

# Final gap score with temporal factor (if enabled)
df_country["gap_score"] = df_country["base_gap_score"] * df_country["temporal_multiplier"]

# Rank
ranked = df_country.sort_values("gap_score", ascending=False).reset_index(drop=True)

final = ranked[[
    "iso3",
    "people_in_need",
    "requirements",
    "funding",
    "coverage_ratio",
    "cumulative_historical_gap",
    "years_with_data",
    "temporal_multiplier",
    "gap_score"
]].copy()

# Ensure consistent data types
final["people_in_need"] = final["people_in_need"].astype('int64')
final["cumulative_historical_gap"] = final["cumulative_historical_gap"].astype('float64')
final["years_with_data"] = final["years_with_data"].astype('int64')

# Convert pandas DataFrame back to Spark and save as a table
# Use overwriteSchema to handle schema changes without needing DROP permissions
final_spark = spark.createDataFrame(final)
final_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.unochadatasets.gap_rankings_2025")

print(f"Data saved to workspace.unochadatasets.gap_rankings_2025")
print(f"  - Crisis Category: {crisis_category}")
print(f"  - Temporal Factor: {include_temporal}")
print(f"  - Demographic Filter: {demographic_filter}")
print(f"  - Total countries ranked: {len(final)}")

if demographic_filter != "All":
    print(f"\nDemographic Filtering:")
    print(f"  - Applied {demographic_filter} population proportions to people_in_need")
    
if include_temporal == "Yes":
    print(f"\nTemporal Factor Stats (based on historical gap scores):")
    print(f"  - Average cumulative historical gap: ${final['cumulative_historical_gap'].mean()/1e9:.2f}B")
    print(f"  - Max cumulative historical gap: ${final['cumulative_historical_gap'].max()/1e9:.2f}B ({final.loc[final['cumulative_historical_gap'].idxmax(), 'iso3']})")
    print(f"  - Average temporal multiplier: {final['temporal_multiplier'].mean():.2f}x")
    print(f"  - Max temporal multiplier: {final['temporal_multiplier'].max():.2f}x")
    print(f"  - Countries with multiplier > 2.0x: {len(final[final['temporal_multiplier'] > 2.0])}")
else:
    print(f"\nTemporal factor disabled - rankings based only on 2025 data")

In [0]:
# Export gap rankings for frontend development
# Write to Unity Catalog Volume (serverless-compatible)

# Use the exports volume
volume_path = "/Volumes/workspace/unochadatasets/exports"

# Export as CSV
csv_path = f"{volume_path}/gap_rankings_2025.csv"
final.to_csv(csv_path, index=False)
print(f"✓ CSV exported to: {csv_path}")

# Export as JSON
json_path = f"{volume_path}/gap_rankings_2025.json"
final.to_json(json_path, orient="records", indent=2)
print(f"✓ JSON exported to: {json_path}")

print(f"\nTo download these files:")
print(f"1. Navigate to Catalog Explorer")
print(f"2. Go to workspace > unochadatasets > exports volume")
print(f"3. Right-click the file and select 'Download'")

In [0]:
# Display top 20 countries by gap score
display(final.head(20))

In [0]:
import json
import time
from pathlib import Path

# All crisis categories
crisis_categories = [
    "ALL", "CCM", "CSS", "EDU", "ERY", "FSC", "HEA", "LOG", "MPC", "MS", 
    "NUT", "PRO", "PRO-CPN", "PRO-GBV", "PRO-HLP", "PRO-MIN", "SHL", "TEL", "WSH"
]

# Temporal options
temporal_options = ["Yes", "No"]

# Demographic options
demographic_options = ["All", "Men", "Women", "Children"]

# Output directory in the GitHub repo
output_dir = Path("/Workspace/Repos/anicernak@gmail.com/UN-OCHA-/data/rankings")
output_dir.mkdir(parents=True, exist_ok=True)

start_time = time.time()
total_combinations = len(crisis_categories) * len(temporal_options) * len(demographic_options)
print(f"Generating rankings for {total_combinations} parameter combinations...")
print(f"  - Crisis categories: {len(crisis_categories)}")
print(f"  - Temporal options: {len(temporal_options)}")
print(f"  - Demographic options: {len(demographic_options)}")
print(f"  - Total files to generate: {total_combinations * 2} (CSV + JSON)\n")

successful = 0
failed = 0

for crisis_cat in crisis_categories:
    for temporal_opt in temporal_options:
        for demographic_opt in demographic_options:
            try:
                # Calculate demographic proportions per country
                if demographic_opt != "All":
                    # Define demographic filters
                    if demographic_opt == "Children":
                        demo_filter = "Age_range IN ('0-4', '5-9', '10-14', '15-19')"
                    elif demographic_opt == "Men":
                        demo_filter = "Gender = 'm' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
                    elif demographic_opt == "Women":
                        demo_filter = "Gender = 'f' AND Age_range NOT IN ('0-4', '5-9', '10-14', '15-19', 'all')"
                    
                    # Get demographic population for selected group
                    demographic_pop = spark.sql(f"""
                    SELECT 
                        ISO3 as iso3,
                        SUM(Population) as demographic_population
                    FROM workspace.unochadatasets.cod_population_admin_0
                    WHERE {demo_filter}
                    GROUP BY ISO3
                    """)
                    
                    # Get total population per country
                    total_pop = spark.sql("""
                    SELECT 
                        ISO3 as iso3,
                        SUM(Population) as total_population
                    FROM workspace.unochadatasets.cod_population_admin_0
                    WHERE Age_range = 'all' AND Gender = 'all'
                    GROUP BY ISO3
                    """)
                    
                    # Calculate proportion
                    demo_proportions_spark = demographic_pop.join(total_pop, "iso3", "inner")
                    demo_proportions = demo_proportions_spark.toPandas()
                    demo_proportions["demographic_proportion"] = (
                        demo_proportions["demographic_population"] / demo_proportions["total_population"]
                    )
                    demo_proportions = demo_proportions[["iso3", "demographic_proportion"]]
                else:
                    demo_proportions = pd.DataFrame(columns=["iso3", "demographic_proportion"])
                
                # FIX: HNO baseline - AGGREGATE by country to avoid duplicates
                hno_base_spark = spark.sql(f"""
                SELECT
                  `Country ISO3` AS iso3,
                  SUM(CAST(`In Need` AS BIGINT)) AS people_in_need
                FROM workspace.unochadatasets.hpc_hno_2025
                WHERE `Cluster` = '{crisis_cat}'
                  AND (`Category` IS NULL OR TRIM(`Category`) = '')
                GROUP BY `Country ISO3`
                """)
                
                hno_base = hno_base_spark.toPandas()
                hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
                hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])
                
                # Apply demographic proportion to people_in_need
                if demographic_opt != "All":
                    hno_base = hno_base.merge(demo_proportions, on="iso3", how="left")
                    hno_base["demographic_proportion"] = hno_base["demographic_proportion"].fillna(0.5)
                    hno_base["people_in_need"] = (hno_base["people_in_need"] * hno_base["demographic_proportion"]).astype('int64')
                    hno_base = hno_base[["iso3", "people_in_need"]]
                
                # FTS baseline
                fts_base_spark = spark.sql("""
                SELECT
                  `countryCode` AS iso3,
                  SUM(`requirements`) AS requirements,
                  SUM(`funding`) AS funding
                FROM workspace.unochadatasets.fts_requirements_funding_global
                WHERE `year` = 2025
                GROUP BY `countryCode`
                """)
                
                fts_base = fts_base_spark.toPandas()
                fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
                fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
                fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])
                
                # Temporal factor calculation (only if enabled)
                if temporal_opt == "Yes":
                    fts_historical_spark = spark.sql("""
                    SELECT
                      `countryCode` AS iso3,
                      `year`,
                      SUM(`requirements`) AS requirements,
                      SUM(`funding`) AS funding
                    FROM workspace.unochadatasets.fts_requirements_funding_global
                    WHERE `year` BETWEEN 2020 AND 2024
                    GROUP BY `countryCode`, `year`
                    """)
                    
                    fts_historical = fts_historical_spark.toPandas()
                    fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
                    fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
                    fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
                    fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
                    
                    fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
                    fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
                    
                    historical_gaps = fts_historical.groupby("iso3")["gap_score_year"].sum().reset_index()
                    historical_gaps.columns = ["iso3", "cumulative_historical_gap"]
                    
                    years_count = fts_historical.groupby("iso3")["year"].count().reset_index()
                    years_count.columns = ["iso3", "years_with_data"]
                    
                    temporal_factor = historical_gaps.merge(years_count, on="iso3", how="left")
                else:
                    temporal_factor = pd.DataFrame(columns=["iso3", "cumulative_historical_gap", "years_with_data"])
                
                # Merge
                df = hno_base.merge(fts_base, on="iso3", how="inner")
                df = df.merge(temporal_factor, on="iso3", how="left")
                df["cumulative_historical_gap"] = df["cumulative_historical_gap"].fillna(0).astype('float64')
                df["years_with_data"] = df["years_with_data"].fillna(0).astype('int64')
                
                df = df[(df["people_in_need"] > 0) & (df["requirements"] > 0)].copy()
                df_country = df.copy()
                
                # Compute baseline
                df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"]
                df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1)
                df_country["base_gap_score"] = df_country["people_in_need"] * (1 - df_country["coverage_ratio"])
                
                # Apply temporal multiplier
                if temporal_opt == "Yes":
                    df_country["avg_annual_historical_gap"] = df_country["cumulative_historical_gap"] / df_country["years_with_data"].replace(0, 1)
                    TEMPORAL_WEIGHT = 0.5
                    df_country["temporal_multiplier"] = 1 + (df_country["avg_annual_historical_gap"] / df_country["base_gap_score"].replace(0, 1)) * TEMPORAL_WEIGHT
                    df_country["temporal_multiplier"] = df_country["temporal_multiplier"].clip(lower=1.0, upper=3.0)
                else:
                    df_country["temporal_multiplier"] = 1.0
                    df_country["avg_annual_historical_gap"] = 0.0
                
                df_country["gap_score"] = df_country["base_gap_score"] * df_country["temporal_multiplier"]
                
                # Rank
                ranked = df_country.sort_values("gap_score", ascending=False).reset_index(drop=True)
                
                final = ranked[[
                    "iso3",
                    "people_in_need",
                    "requirements",
                    "funding",
                    "coverage_ratio",
                    "cumulative_historical_gap",
                    "years_with_data",
                    "temporal_multiplier",
                    "gap_score"
                ]].copy()
                
                # Ensure consistent data types
                final["people_in_need"] = final["people_in_need"].astype('int64')
                final["cumulative_historical_gap"] = final["cumulative_historical_gap"].astype('float64')
                final["years_with_data"] = final["years_with_data"].astype('int64')
                
                # Add country names
                country_lookup_spark = spark.table("workspace.unochadatasets.country_iso3_lookup")
                country_lookup = country_lookup_spark.toPandas()
                final = final.merge(country_lookup, on="iso3", how="left")
                final["country_name"] = final["country_name"].fillna(final["iso3"])
                
                # Reorder columns to put country_name after iso3
                cols = ["iso3", "country_name"] + [col for col in final.columns if col not in ["iso3", "country_name"]]
                final = final[cols]
                
                # Generate file suffix
                temporal_suffix = "with_temporal" if temporal_opt == "Yes" else "no_temporal"
                demographic_suffix = demographic_opt.lower()
                
                # Save CSV
                csv_filename = f"gap_rankings_{crisis_cat.lower()}_{temporal_suffix}_{demographic_suffix}.csv"
                csv_path = output_dir / csv_filename
                final.to_csv(str(csv_path), index=False)
                
                # Save JSON
                json_filename = f"gap_rankings_{crisis_cat.lower()}_{temporal_suffix}_{demographic_suffix}.json"
                json_path = output_dir / json_filename
                final_json = final.to_dict(orient="records")
                with open(str(json_path), "w") as f:
                    json.dump(final_json, f, indent=2)
                
                successful += 1
                
            except Exception as e:
                print(f"  ✗ Failed: {crisis_cat}, temporal={temporal_opt}, demographic={demographic_opt}")
                print(f"    Error: {str(e)}")
                failed += 1

elapsed_time = time.time() - start_time

print(f"\n{'='*60}")
print(f"Generation complete!")
print(f"  - Successful: {successful}/{total_combinations}")
print(f"  - Failed: {failed}/{total_combinations}")
print(f"  - Files generated: {successful * 2} (CSV + JSON)")
print(f"  - Output directory: {output_dir}")
print(f"  - Execution time: {elapsed_time:.1f} seconds")
print(f"{'='*60}")